# Segmentation and motion correction of in-vivo recordings - efferent data

Before running this notebook, generate an ROI called `correctionReference.npy` using [Robopy](https://github.com/fedeceri85/robopy2). This ROI should be placed in the same folder as the recording and contain a trace where the "jumps" are visible as decreases (ideally) or increases in fluorescence.

This notebook contains the preprocessing steps and segmentation routines for in vivo recordings. Some utilities used in this notebook are:

- `thorlabsFile` from `movieTools`: This class contains the actual recording data, functions to smooth and filter it, and display it in a Napari viewer. The `thorlabsFile` object is passed to other routines to extract data. When loading files, they are automatically smoothed with a 3D Gaussian kernel (2x2x2).
- `jumpFramesFinder` from `visualisationTools`: This displays an ipywidget interface with tools to remove out-of-focus frames. This interface displays the `correctionReference` generated using Robopy (first step of the analysis).
- `jupyterpy` from `movieTools`: This interface connects to the currently displayed `thorlabsFile` and contains buttons for the segmentation of hair cells, calcium waves, and fibers.
- `maskMatching` from `movieTools`: This is an ipywidgets/Napari interface that combines repetitive recordings of the same field of view to generate a `matchingMask`, i.e., a label image where hair cell ROIs are color-coded with the same color.


In [1]:
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))


In [2]:
from IPython.display import display, HTML
display(HTML("<style>.jp-Cell { margin-left: -50% !important; margin-right: -50% !important; }</style>"))

In [3]:
%pylab
%load_ext autoreload
%autoreload 2
%matplotlib inline
import pandas as pd
import os
import sys
sys.path.append('../../src')
import visualisationTools as vu
import traceUtilities as tu
import ast 
from movieTools import thorlabsFile

parameterFolder = '../../parameters/'
fileHeader = 'RGECO'

inputFilename = os.path.join('../../',fileHeader)+'.xlsx' #List of recordings with metadata in an excel file. This is the input file generated by the user.
corrFilename = os.path.join(parameterFolder,fileHeader)+ '_with_corr_params.csv'  #List of recordings with metadata and correction parameters used in jumpFramesFinder (frame intervals to discard and findpeaks parameters.). This is generated by this notebook to 
                                                                                    #keep track of the parameters.

jumpFrameFilename = os.path.join(parameterFolder,fileHeader+'jump_frames.csv') # List of minima found in the correctionReference trace with the findPeaks method
jumpFrameMaxFilename = os.path.join(parameterFolder,fileHeader+'jumpMax_frames.csv') # List of maxima found in the correctionReference trace  with the findPeaks method
correctionReferenceTraceFile = os.path.join(parameterFolder,fileHeader+'correctionReference.csv') # Correction Reference list. 

Using matplotlib backend: module://matplotlib_inline.backend_inline
%pylab is deprecated, use %matplotlib inline and import the required libraries.
Populating the interactive namespace from numpy and matplotlib


First we load all the excel files above as pandas dataframes, or create an empty dataframe if they don't exist

In [4]:
alldata = pd.read_excel(inputFilename)
alldata = alldata[alldata['discard']!=1]

try:
    corr_param = pd.read_csv(corrFilename)
    #Change the first drive to match the one of the alldata file
    drive = alldata['Folder'].iloc[0][0]
    corr_param['Folder'] = drive +corr_param.loc[:,'Folder'].str[1:]    
    
except:
    print('No jump correction parameter file')

try:
    allminima = pd.read_csv(jumpFrameFilename)
except FileNotFoundError:
    allminima = pd.DataFrame()

try:
    allmaxima = pd.read_csv(jumpFrameMaxFilename)
except FileNotFoundError:
    allmaxima = pd.DataFrame()


try:
    correctionReferenceTrace = pd.read_csv(correctionReferenceTraceFile)
except FileNotFoundError:
    correctionReferenceTrace = pd.DataFrame()

No jump correction parameter file


Then we go through the list of recordings and compare it to the list of parameters. We copy over the already calculated parameters for all the files that have already been processed, and use default (blank) values for all the new files.

In [5]:
for el in alldata['Folder'].unique():
    try:
        alldata.loc[alldata['Folder']==el,'Window left'] = corr_param.loc[corr_param['Folder']==el,'Window left'].values[0]
        alldata.loc[alldata['Folder']==el,'Window right'] = corr_param.loc[corr_param['Folder']==el,'Window right'].values[0]
        alldata.loc[alldata['Folder']==el,'Minima order'] = corr_param.loc[corr_param['Folder']==el,'Minima order'].values[0]



    except (IndexError, NameError, KeyError):
        print('Could not find correction parameters for '+ el+' Setting to default.')
        alldata.loc[alldata['Folder']==el,'Window left'] = 0
        alldata.loc[alldata['Folder']==el,'Window right'] = 0
        alldata.loc[alldata['Folder']==el,'Minima order'] = 50 



    try:
        alldata.loc[alldata['Folder']==el,'Window Max left'] = corr_param.loc[corr_param['Folder']==el,'Window Max left'].values[0]
        alldata.loc[alldata['Folder']==el,'Window Max right'] = corr_param.loc[corr_param['Folder']==el,'Window Max right'].values[0]
        alldata.loc[alldata['Folder']==el,'Maxima order'] = corr_param.loc[corr_param['Folder']==el,'Maxima order'].values[0]
    except (IndexError, NameError, KeyError):
        alldata.loc[alldata['Folder']==el,'Window Max left'] = 0
        alldata.loc[alldata['Folder']==el,'Window Max right'] = 0
        alldata.loc[alldata['Folder']==el,'Maxima order'] = 50
        
    try:
        alldata.loc[alldata['Folder']==el,'SmoothOrder'] = corr_param.loc[corr_param['Folder']==el,'SmoothOrder'].values[0]
    except (IndexError, NameError, KeyError):
        alldata.loc[alldata['Folder']==el,'SmoothOrder'] = 11
        
alldata.loc[alldata['Window left'].isna(),'Window left'] = 0
alldata.loc[alldata['Window right'].isna(),'Window right'] = 0
alldata.loc[alldata['Minima order'].isna(),'Minima order'] = 50

alldata.loc[alldata['Folder'].isna(),'Window Max left'] = 0
alldata.loc[alldata['Folder'].isna(),'Window Max right'] = 0
alldata.loc[alldata['Folder'].isna(),'Maxima order'] = 50

alldata['Window left'] = alldata['Window left'].astype(int)
alldata['Window right'] = alldata['Window right'].astype(int)
alldata['Minima order'] = alldata['Minima order'].astype(int)

alldata['Window Max left'] = alldata['Window Max left'].astype(int)
alldata['Window Max right'] = alldata['Window Max right'].astype(int)
alldata['Maxima order'] = alldata['Maxima order'].astype(int)


alldata.loc[alldata['Minima order']==1,'Minima order'] = 50
alldata.loc[alldata['Maxima order']==1,'Maxima order'] = 50



alldata.loc[alldata['Folder'].isna(),'SmoothOrder'] = 11
alldata['SmoothOrder'] = alldata['SmoothOrder'].astype(int)


try:
    for el in alldata['Folder'].unique():
        alldata.loc[alldata['Folder']==el,'ExtraCorrectionIntervals'] = corr_param.loc[corr_param['Folder']==el,'ExtraCorrectionIntervals'].values[0]
except:
    pass

for j,el in alldata.iterrows():
    try:
        alldata.at[j,'ExtraCorrectionIntervals'] = ast.literal_eval(alldata.at[j,'ExtraCorrectionIntervals'] )
    except (ValueError,KeyError):
         alldata.at[j,'ExtraCorrectionIntervals'] = np.nan
            
alldata['ExtraCorrectionIntervals'] =  alldata['ExtraCorrectionIntervals'].astype('object')



try:
    for el in alldata['Folder'].unique():
        alldata.loc[alldata['Folder']==el,'TemplateIntervals'] = corr_param.loc[corr_param['Folder']==el,'TemplateIntervals'].values[0]
except:
    pass

for j,el in alldata.iterrows():
    try:
        alldata.at[j,'TemplateIntervals'] = ast.literal_eval(alldata.at[j,'TemplateIntervals'] )
    except (ValueError,KeyError):
         alldata.at[j,'TemplateIntervals'] = np.nan

alldata['TemplateIntervals'] =  alldata['TemplateIntervals'].astype('object')

Could not find correction parameters for Z:\Users\Char\2PI Data\2025\2025_11_04_TMEM16A-Pax2Cre-GCaMP6f_PHPebMyo15CreRGECO AAV_in vivo_P8_Tattoo 1\1_001 Setting to default.


In [6]:
master = alldata.copy().reset_index()

# 1- Jump Correction. 
Remember to run the block after this every now and then to save the results in case this notebook crashes
`jumpFramesFinder` is a comprehensive tool that combines interactive widgets, data processing, and visualization to assist users in identifying and correcting frame jumps in their datasets.
The jumpFramesFinder function is designed to facilitate the identification and correction of frame jumps in a in vivo movie. This function utilizes a variety of widgets from the `ipywidgets` library to create an interactive user interface  that allows users to adjust parameters and visualize the effects of these adjustments in real-time.

The user can utilise various sliders to control various parameters such as the trace number, window sizes for minima and maxima detection, and the order of minima and maxima. These sliders allow users to fine-tune the detection of frame jumps by adjusting the sensitivity and range of the detection algorithm.

Users can perform specific actions, such as loading original or corrected movie, saving processed files, and creating templates for further analysis.

The function also sets up a plot using `plotly.graph_objects` to visualize the original and corrected traces, as well as the detected jumps. This visualization helps users to see the impact of their adjustments and make decisions about the parameters they set.

The corrected movie is visualised through the `thorlabsFile` object in a napari window.

After this step, launch the `batchModtionCorrect.ipynb` notebook to motion correct the "jumpCorrected.tif" files, which will be saved as `jumpCorrected-mc.tif` files. After that, come back to this notebook, and run this block to load the motion corrected files, to be used for segmentation.



In [7]:
tb = thorlabsFile()
vu.jumpFramesFinder(master,allminima,allmaxima,correctionReferenceTrace,tb)

Output()

In [8]:
data1 = tb.app.layers['Image'].data
data2 = tb.app.layers['Image channel 2'].data

In [9]:
# determine the scaling factor between the two channels
argmaxInd = (data1.astype(np.float32)-data1[:100,:,:].mean(0)).argmax(0)

scaling_factor1 = data1[argmaxInd, np.arange(data1.shape[1])[:,None], np.arange(data1.shape[2])[None,:]] - data1[:100,:,:].mean(0)


scaling_factor2 = data2[argmaxInd, np.arange(data2.shape[1])[:,None], np.arange(data2.shape[2])[None,:]] - data2[:100,:,:].mean(0)






In [10]:
scaling_factor = scaling_factor2 / scaling_factor1



# scaling_factor[scaling_factor==np.inf] = 0.00001
# scaling_factor[np.isnan(scaling_factor)] = 0.000001

# scaling_factor[scaling_factor>1]=1
scaling_factor[scaling_factor<0]=0

In [11]:
# explicit broadcasting
data2_scaled = np.zeros((data2.shape[0], data2.shape[1], data2.shape[2]))
background = data1[:100,:,:].mean(0)
background2 = data2[:100,:,:].mean(0)
for i in range(data2.shape[0]):
    data2_scaled[i,:,:] = data2[i,:,:].astype(np.float32) - ((data1[i,:,:].astype(np.float32) - background) * scaling_factor )


data2_scaled[data2_scaled<0] = 0    
data2_scaled = data2_scaled.astype(np.uint16)

In [26]:
tb.app.add_image(data2_scaled, name='Image channel 2 scaled', colormap='red', blending='additive')

AttributeError: 'Image' object has no attribute 'type'

In [ ]:

master.to_csv(corrFilename)
allminima.to_csv(jumpFrameFilename)
allmaxima.to_csv(jumpFrameMaxFilename)

correctionReferenceTrace.to_csv(correctionReferenceTraceFile)

## The next block displays the jupyterpy interface. This connects to the opened image in napari (the current thorlabsFile object, it should be loaded using the jumpFramesFinder above) and provides buttons with specific analysis steps for each type of experiment.  

The jupyterPy function is designed to create an interactive user interface (UI) for analyzing image data within a Jupyter notebook. This function utilizes the ipywidgets library to create various interactive widgets, such as buttons, sliders, and text inputs, which are then organized into a tabbed layout for different analysis tasks.
The istance of jupyterPy connects to the movie displayed in the existing napari windows (passed through the thorlabsFile object.)

The software allows to perform specific segmentation analysis for IHCs, Calcium waves, and afferent terminals. 
For IHCs, the software uses cellpose to automatically identify the cell bodies. The segmentation is shown as a `label` layer (named 'Masks') in Napari. Users can modify the existing ROIs using Napari tools (e.g, splitting ROIs). The function also allows to plot and save the fluorescence profile of the individual ROIs. 


In [23]:
tb.app.layers['Image']._type_string

'image'

In [30]:
from movieTools import jupyterPy
JP = jupyterPy(tb)

IndexError: too many indices for array: array is 2-dimensional, but 3 were indexed

# 2- Match ROIs from different recordings

In [ ]:
#Change drive 
localDrive = 'D'
master['Folder'] = localDrive + master['Folder'].str[1:]

In [ ]:
from movieTools import maskMatching,extractImagesMaskMatching

In [ ]:
maskMatching(master)

# 3- Calculate the local pixel correlation
This block calculates how much the pixels belonging to the same ROI are correlated. This helps with peaks identification, as peaks resulting from artifacts (e.g., cross-talk from a neighboring cell) will have a low correlation and can be rejected on this basis
The correlation traces are saved in the same folder as the traces (processedMovies folder inside the experiment folder)

In [ ]:
for j,el in master.iterrows():
    
    if not os.path.exists(os.path.join(el['Folder'],'processedMovies','traces_localCorr.csv')):
        print(el['Folder'])
        print(j)
        df = tu.calculatePixelRollingCorr(el['Folder'],int(el['fps']/2),8,addNoise=True)
        df.to_csv(os.path.join(el['Folder'],'processedMovies','traces_localCorr.csv'))

# N numbers

In [ ]:
master.groupby(['Age','Mouse ID']).count()['Folder']